In [ ]:
!pip install enoslib ipywidgets --break-system-packages

In [ ]:
!ssh rennes.grid5000.fr hostname

In [ ]:
!ssh-add ~/.ssh/id_ed25519

### G5K connection

Setup the `.python-grid5000.yaml` file with the username and password used to login to grid5000.

The file should look like this:
```yaml
username: G5K_LOGIN
password: G5K_password
```

-> required in order for the enoslib calls to g5k to work.

In addition to this, make sure to add these lines to your ssh configuration:

```text
Host g5k
    User G5K_LOGIN
    HostName access.grid5000.fr
    ForwardAgent no

Host !access.grid5000.fr *.grid5000.fr
    User G5K_LOGIN
    ProxyJump G5K_LOGIN@access.grid5000.fr
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host access.grid5000.fr
    User G5K_LOGIN
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host *.g5k
    User G5K_LOGIN
    ProxyCommand ssh g5k -W "$(basename %h .g5k):%p"
    ForwardAgent no
```

Make sure to replace `G5K_LOGIN` with your g5k username (the one from the site)

In [ ]:
from grid5000 import Grid5000
import enoslib as en
import logging
import os
from datetime import datetime, timedelta

conf_file = os.path.join(os.environ.get("HOME"), ".python-grid5000.yaml") # type: ignore
gk = Grid5000.from_yaml(conf_file)

# map each cluster to its site
cluster_to_site = {}
for site in gk.sites.list():
    for cluster in site.clusters.list():
        cluster_to_site[cluster.uid] = site.uid

# JOB CONFIGURATION
JOB_NAME="fcquic_emulated_relay_eval"
CLUSTER="chirop"
JOB_WALLTIME=timedelta(hours=4, minutes=30)

# usage policy check: the job cannot cross the day to night boundary at 7pm if it was submitted before 5 pm. Any job started after 5 pm can cross the boundary
# I had the issue once so this check is there to avoid receiving a usage policy violation email...
datetime_now = datetime.now()
job_end_dt = datetime_now + JOB_WALLTIME
if datetime_now.hour <= 17 and job_end_dt.hour >= 19 and datetime_now.weekday() <= 5: # only check during weekdays
    raise RuntimeError("This job reservation will violate the usage policy and will cross the day night boundary")

# Display some general information about the library
en.check()
# Enable rich logging
_ = en.init_logging()

conf = (
    en.G5kConf.from_settings(
        job_name=JOB_NAME,
        walltime=str(JOB_WALLTIME),
        env_name="debian12-big",
        job_type=["deploy"],
    )
    .add_machine(
        roles=["server"],
        servers=["chirop-5.lille.grid5000.fr"]
    ) 
)

# This will validate the configuration, but not reserve resources yet
provider = en.G5k(conf)

In [ ]:
print("Reserving resources...")

# Get actual resources
roles, networks = provider.init()
display(roles)
display(networks)

# Fill in network information from nodes
roles = en.sync_info(roles, networks)

with en.actions(roles=roles, gather_facts=True) as a:
    a.apt(
        task_name="Install packages",
        name=["tcpdump", "cmake", "clang", "python3.13-venv", "python-is-python3", "python3-pip", "btop", "htop"],
        state="present",
    )

    # install frr
    a.file(
        task_name="Ensure apt keyring directory exists",
        path="/usr/share/keyrings",
        state="directory",
        mode="0755",
    )
    a.get_url(
        task_name="Download FRR GPG key",
        url="https://deb.frrouting.org/frr/keys.gpg",
        dest="/usr/share/keyrings/frrouting.gpg",
        mode="0644",
    )
    a.apt_repository(
        task_name="Add FRR apt repository",
        repo="deb [signed-by=/usr/share/keyrings/frrouting.gpg] https://deb.frrouting.org/frr {{ ansible_distribution_release }} frr-stable",
        filename="frr",
        state="present",
    )
    a.apt(
        task_name="Install FRR packages",
        name=["frr", "frr-pythontools"],
        state="present",
        update_cache=True,
    )

    # rust
    a.get_url(
        task_name="Download rustup installer",
        url="https://sh.rustup.rs",
        dest="/tmp/rustup-init.sh",
        mode="0755",
    )
    a.shell(
        task_name="Install Rust stable (rustup)",
        cmd="sh /tmp/rustup-init.sh -y --default-toolchain stable",
        creates="/root/.cargo/bin/rustup",
    )
    a.shell(
        task_name="Install Rust nightly toolchain",
        cmd="/root/.cargo/bin/rustup toolchain install nightly",
    )

    results = a.results


In [ ]:
# print os and kernel versions
from enoslib.api import Results

res: Results = en.run_command("uname -a", roles=roles)
print([res.stdout for res in res])


### Uploading project with SCP 

In [ ]:
import subprocess

local_bin_dir = f"/home/corentin/fcquic_applications_master_thesis"
remote_bin_dir = "/tmp"

for role, nodes in roles.items():
    for node in nodes:
        host = node.address
        print(f"pushing binary to {host} (role: {role})")
    
        subprocess.run(["ssh", host, f"mkdir -p {remote_bin_dir}/fcquic_relay"], check=True)
        subprocess.run(["ssh", host, f"mkdir -p {remote_bin_dir}/evaluations"], check=True)
        subprocess.run(["ssh", host, f"mkdir -p {remote_bin_dir}/multicast-quic"], check=True)
        
        subprocess.run(["rsync", "-az",
                        "--exclude=logs/",
                        "--exclude=target/",
                        f"{local_bin_dir}/fcquic_chat/", f"{host}:{remote_bin_dir}/fcquic_chat/"], check=True)
        subprocess.run(["rsync", "-az",
                        "--exclude=target/",
                        f"{local_bin_dir}/multicast-quic/", f"{host}:{remote_bin_dir}/multicast-quic/"], check=True)
        subprocess.run(["rsync", "-az", "--exclude=venv/", f"{local_bin_dir}/evaluations/", f"{host}:{remote_bin_dir}/evaluations/"], check=True)
        
print("pushed project to node")


### Running the experiment

In [ ]:

en.run_command(f"pip install graphviz networkx pandas brokenaxes --break-system-packages", roles=roles)

*Important:* SSH into the machine `root@vianden-1.luxembourg.grid5000.fr`, then run the following commands:
- `cd /home/cdetry/chat`
- `python -m venv venv`
- `source ./venv/bin/activate`
- `pip install npf`

In [ ]:
en.run_command(f"cd {remote_bin_dir}/fcquic_relay && cargo build --release", roles=roles)

In [ ]:
# TODO
# for role, nodes in roles.items():
#     for node in nodes:
#         host = node.address
#         subprocess.run(
#             ["ssh", "root@"+host, f"cd {remote_bin_dir}/evaluations/ && bash run_test.sh {TEST_DIR_NAME} {TOPO_CONF_NAME} {USE_POISSON}"],
#             check=True,
#         )


In [ ]:
import subprocess
import os

result_filename = f"npf_out_local_relay_eval.csv"

remote_out_dir = f"{remote_bin_dir}/evaluations/tests/local_relay_eval/out/"
local_out_dir = f"{local_bin_dir}/evaluations/tests/local_relay_eval/out/"

os.makedirs(local_out_dir, exist_ok=True)

for role, nodes in roles.items():
    for node in nodes:
        host = node.address
        print(f"downloading results from {host}")
        subprocess.run(
            ["rsync", "-az", f"root@{host}:{remote_out_dir}{result_filename}", local_out_dir],
            check=True,
        )

print(f"results saved to {local_out_dir}{result_filename}")


In [ ]:
import subprocess

result_filename = f"npf_out_local_relay_eval"
graphs_dir = f"{local_bin_dir}/evaluations/graphs"
csv_path = f"{local_bin_dir}/evaluations/tests/local_relay_eval/out/{result_filename}"
output_dir = f"{graphs_dir}/local_relay_eval"

os.makedirs(output_dir, exist_ok=True)

subprocess.run(
    ["python", f"{graphs_dir}/local_relay_graphs.py", csv_path, output_dir],
    check=True,
)

print(f"graphs written to {output_dir}")


### Stopping the current booking

In [ ]:
provider.destroy()